<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/05_cargar_embeddings_pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este notebook voy a cargar en Pinecone los embeddings que generé en
el Notebook 04.

Cada vector representa un chunk de los documentos oficiales de ARCA.

Además del vector, voy a guardar metadatos que me permitirán recuperar
posteriormente el texto original, el documento de origen y la URL.

El objetivo de este notebook es dejar preparada la base vectorial para
realizar búsquedas semánticas en el sistema RAG.

## Montaje de Google Drive

En esta celda monto Google Drive para acceder al archivo de embeddings
generado en el Notebook 04.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Definición de las rutas

En esta celda defino la carpeta del proyecto y la ubicación del archivo
que contiene los embeddings.

Utilizo la misma estructura de carpetas que en los notebooks anteriores
para mantener el proyecto organizado.

In [2]:
from pathlib import Path

CARPETA_PROYECTO = Path(
    "/content/drive/MyDrive/TP_RAG_ARCA"
)

CARPETA_CORPUS = (
    CARPETA_PROYECTO / "corpus"
)

ARCHIVO_EMBEDDINGS = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo_embeddings.json"
)

print("Archivo de embeddings:")
print(ARCHIVO_EMBEDDINGS)

Archivo de embeddings:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_embeddings.json


## Verificación del archivo de embeddings

En esta celda verifico que el archivo generado en el Notebook 04 exista
antes de intentar cargarlo en Pinecone.

In [3]:
if not ARCHIVO_EMBEDDINGS.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n"
        f"{ARCHIVO_EMBEDDINGS}"
    )

print("Archivo encontrado correctamente.")

print(
    f"Tamaño: "
    f"{ARCHIVO_EMBEDDINGS.stat().st_size:,} bytes"
)

Archivo encontrado correctamente.
Tamaño: 507,042 bytes


## Instalación de Pinecone

En esta celda instalo la librería oficial de Pinecone que voy a utilizar
para conectarme con la base vectorial.

In [4]:
!pip install -q pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 23.4 MB/s eta 0:00:00


## Importación de librerías

En esta celda importo las librerías que voy a utilizar.

Utilizo `json` para cargar los embeddings y `Pinecone` para conectarme
con el índice vectorial.

In [5]:
import json

from pinecone import Pinecone
from google.colab import userdata

## Carga de la API key de Pinecone

En esta celda recupero la API key desde los Secrets de Google Colab.

No guardo la clave dentro del notebook ni dentro de GitHub.

In [6]:
PINECONE_API_KEY = userdata.get(
    "PINECONE_API_KEY"
)

if not PINECONE_API_KEY:
    raise ValueError(
        "No se encontró PINECONE_API_KEY "
        "en los Secrets de Google Colab."
    )

print("API key de Pinecone cargada correctamente.")

API key de Pinecone cargada correctamente.


## Carga del corpus con embeddings

En esta celda cargo el archivo generado en el Notebook 04.

Cada elemento contiene el texto de un chunk, sus metadatos y su embedding.

In [7]:
with open(
    ARCHIVO_EMBEDDINGS,
    "r",
    encoding="utf-8"
) as archivo:

    chunks = json.load(archivo)

print(
    f"Chunks cargados: {len(chunks)}"
)

Chunks cargados: 43


## Verificación de los embeddings

En esta celda verifico que los 43 chunks tengan un embedding y que todos
los vectores tengan la misma dimensión.

Esta dimensión debe coincidir con la dimensión configurada en el índice
de Pinecone.

In [8]:
dimensiones = set()

for chunk in chunks:

    embedding = chunk.get("embedding")

    if embedding is None:
        raise ValueError(
            f"El chunk {chunk.get('chunk_id')} "
            "no tiene embedding."
        )

    dimensiones.add(
        len(embedding)
    )

print(
    f"Cantidad de chunks: {len(chunks)}"
)

print(
    f"Dimensiones encontradas: {dimensiones}"
)

if len(dimensiones) != 1:
    raise ValueError(
        "Los embeddings no tienen "
        "todos la misma dimensión."
    )

DIMENSION_EMBEDDING = list(
    dimensiones
)[0]

print(
    f"Dimensión utilizada: "
    f"{DIMENSION_EMBEDDING}"
)

Cantidad de chunks: 43
Dimensiones encontradas: {384}
Dimensión utilizada: 384


## Conexión con Pinecone

En esta celda me conecto con Pinecone utilizando la API key almacenada
en los Secrets de Google Colab.

Primero consulto los índices disponibles para verificar que la conexión
funcione correctamente.

In [9]:
pc = Pinecone(
    api_key=PINECONE_API_KEY
)

print("Conexión con Pinecone establecida.")

print("\nÍndices disponibles:")

for indice in pc.list_indexes():
    print(
        f"- {indice['name']}"
    )

Conexión con Pinecone establecida.

Índices disponibles:
- tutorial
- test


## Definición del índice de Pinecone

En esta celda defino el índice específico que voy a utilizar para el TP.

No voy a reutilizar el índice `tutorial` que utilicé durante el tutorial de Pinecone y RAG.

Este índice contendrá exclusivamente los embeddings generados a partir del corpus de documentos oficiales de ARCA sobre Monotributo.

In [10]:
NOMBRE_INDICE = "arca-monotributo"

DIMENSION_EMBEDDINGS = 384
METRIC = "cosine"

print(f"Índice seleccionado: {NOMBRE_INDICE}")
print(f"Dimensión de embeddings: {DIMENSION_EMBEDDINGS}")
print(f"Métrica: {METRIC}")

Índice seleccionado: arca-monotributo
Dimensión de embeddings: 384
Métrica: cosine


## Creación del índice

En esta celda verifico si el índice `arca-monotributo` ya existe.

Si no existe, lo creo utilizando la misma dimensión que tienen mis embeddings y la métrica de similitud coseno.

De esta manera, el índice queda preparado específicamente para almacenar los fragmentos del corpus de ARCA.

In [11]:
from pinecone import ServerlessSpec

indices_existentes = [
    indice["name"]
    for indice in pc.list_indexes()
]

print("Índices existentes:")
for nombre in indices_existentes:
    print(f"- {nombre}")

if NOMBRE_INDICE not in indices_existentes:

    print(f"\nCreando índice '{NOMBRE_INDICE}'...")

    pc.create_index(
        name=NOMBRE_INDICE,
        dimension=DIMENSION_EMBEDDINGS,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    print("Índice creado.")

else:

    print(
        f"\nEl índice '{NOMBRE_INDICE}' ya existe."
    )

Índices existentes:
- tutorial
- test

Creando índice 'arca-monotributo'...
Índice creado.


## Conexión con el índice de ARCA

En esta celda me conecto al índice que acabo de crear.

A partir de este momento voy a utilizar este índice exclusivamente para el proyecto del TP.

In [12]:
index = pc.Index(NOMBRE_INDICE)

print(
    f"Conectado al índice: {NOMBRE_INDICE}"
)

Conectado al índice: arca-monotributo


## Verificación del índice

En esta celda verifico que el índice esté disponible y consulto su estado.

Todavía no contiene vectores porque en las siguientes celdas voy a realizar el proceso de carga de los embeddings.

In [13]:
import time

print("Esperando a que el índice esté listo...")

while True:

    descripcion = pc.describe_index(
        NOMBRE_INDICE
    )

    estado = descripcion["status"]

    print(estado)

    if estado["ready"]:
        break

    time.sleep(2)

print("\nÍndice listo.")

Esperando a que el índice esté listo...
IndexStatus(ready=True, state='Ready')

Índice listo.


## Estado inicial del índice

En esta celda consulto la cantidad de vectores almacenados.

Espero que inicialmente sea cero, ya que todavía no cargué los embeddings del corpus.

In [14]:
estadisticas_indice = index.describe_index_stats()

print(
    estadisticas_indice
)

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


## Preparación de los vectores para Pinecone

En esta celda preparo los chunks generados en el notebook anterior para cargarlos en Pinecone.

Cada chunk ya contiene su embedding, generado en el notebook 04.

Para cada registro voy a conservar el embedding y los metadatos necesarios para poder recuperar posteriormente el texto original.

In [15]:
print(f"Cantidad de chunks: {len(chunks)}")

print(
    f"Cantidad de chunks con embedding: "
    f"{sum('embedding' in chunk for chunk in chunks)}"
)

Cantidad de chunks: 43
Cantidad de chunks con embedding: 43


## Construcción de los registros vectoriales

En esta celda construyo los registros que voy a enviar a Pinecone.

Cada registro tiene:

- un identificador único;
- el embedding generado para el chunk;
- el texto del chunk;
- los metadatos del documento de origen.

De esta manera Pinecone podrá devolverme tanto la similitud vectorial como la información necesaria para reconstruir el contexto del RAG.

In [16]:
vectores = []

for i, chunk in enumerate(chunks):

    vector = {
        "id": f"arca-{i}",
        "values": chunk["embedding"],
        "metadata": {
            "texto": chunk["texto"],
            "archivo": chunk.get(
                "archivo",
                "corpus_arca_monotributo.json"
            ),
            "documento_id": chunk.get(
                "documento_id",
                ""
            ),
            "titulo": chunk.get(
                "titulo",
                ""
            ),
            "chunk": i
        }
    }

    vectores.append(vector)

print(
    f"Vectores preparados: {len(vectores)}"
)

Vectores preparados: 43


## Verificación de un vector

En esta celda inspecciono el primer vector antes de cargarlo en Pinecone.

Quiero verificar que el embedding tenga la dimensión esperada y que los metadatos contengan el texto del chunk correspondiente.

In [17]:
primer_vector = vectores[0]

print("ID:")
print(primer_vector["id"])

print("\nDimensión del embedding:")
print(len(primer_vector["values"]))

print("\nTexto:")
print(primer_vector["metadata"]["texto"][:500])

print("\nMetadatos:")
print(primer_vector["metadata"])

ID:
arca-0

Dimensión del embedding:
384

Texto:
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegación y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la información sobre cómo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
Menú de contenidos
Qué es
INSCRIPCIÓN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal Electrónico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo


Metadatos:
{'texto': 'Inicio - Ayuda sobre el monotributo - Monotributo | ARCA\nEvitar las herramientas de navegación y pasar al contenido\nMonotributo\nMenu\nInicio\nAyuda\nInicio\nAyuda sobre el monotributo\nInicio\nAyuda sobre el monotributo\nToda la información sobre cómo darte de alta y hacer operaciones como monotributista.\nIngresar con clave fiscal\nMenú de contenidos\nQué es\nINSCRIPCIÓN\nInicio\nClave fiscal\nCUIT\nDomicilio Fiscal Ele

## Carga de los embeddings en Pinecone

En esta celda cargo los vectores en mi índice de Pinecone.

Hasta este momento el índice existe, pero está vacío. Con esta operación incorporo los 43 embeddings correspondientes a los chunks del corpus de ARCA.

In [18]:
print(
    f"Cargando {len(vectores)} vectores en Pinecone..."
)

respuesta = index.upsert(
    vectors=vectores
)

print("Carga completada.")
print(respuesta)

Cargando 43 vectores en Pinecone...
Carga completada.
UpsertResponse(upserted_count=43)


## Verificación de la base vectorial

En esta celda consulto nuevamente las estadísticas del índice.

Espero encontrar 43 vectores almacenados en Pinecone.

Esta comprobación me permite asegurar que la base vectorial fue cargada correctamente antes de comenzar con el retrieval.

In [19]:
estadisticas_indice = index.describe_index_stats()

print(estadisticas_indice)

DescribeIndexStatsResponse(dimension=384, total_vector_count=43, metric='cosine', namespaces=1)


## Verificación de la cantidad de vectores

En esta celda comparo la cantidad de embeddings que generé con la cantidad de vectores almacenados en Pinecone.

Si ambas cantidades coinciden, considero terminada la etapa de carga de la base vectorial.

In [20]:
cantidad_embeddings = len(chunks)

cantidad_pinecone = (
    estadisticas_indice["total_vector_count"]
)

print(
    f"Embeddings generados: {cantidad_embeddings}"
)

print(
    f"Vectores en Pinecone: {cantidad_pinecone}"
)

if cantidad_pinecone == cantidad_embeddings:
    print(
        "\nOK: todos los embeddings fueron "
        "cargados correctamente en Pinecone."
    )
else:
    print(
        "\nATENCIÓN: las cantidades no coinciden."
    )

Embeddings generados: 43
Vectores en Pinecone: 43

OK: todos los embeddings fueron cargados correctamente en Pinecone.
